In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
device

'cuda'

In [3]:
import os
import json
import shutil
import pickle
import joblib

import scipy.io

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import optuna
from tqdm.auto import tqdm

import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import optuna
from sklearn.model_selection import train_test_split

/home/usuario-utp/anaconda3/envs/yessica/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# =========================================================
# 1) SEED
# =========================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# =========================================================
# 2) DATASET
# =========================================================
class EEGDataset(Dataset):
    def __init__(self, X, y):
        """
        X: np.ndarray [N, C, T]
        y: np.ndarray [N]
        """
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# =========================================================
# 3) METRICAS
# =========================================================
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred)

    return {
        "acc": acc,
        "bacc": bacc,
        "f1": f1,
        "auc": auc,
        "cm": cm
    }


# =========================================================
# 4) EVALUAR
# =========================================================
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()

    all_y = []
    all_prob = []
    total_loss = 0.0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        out = model(xb, return_dict=True)
        logits = out["logits"]
        probs = out["probs"]

        loss = F.binary_cross_entropy_with_logits(logits, yb)

        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_n += bs

        all_y.append(yb.detach().cpu().numpy())
        all_prob.append(probs.detach().cpu().numpy())

    y_true = np.concatenate(all_y)
    y_prob = np.concatenate(all_prob)

    metrics = compute_metrics(y_true, y_prob)
    metrics["loss"] = total_loss / max(total_n, 1)

    return metrics


# =========================================================
# 5) ENTRENAR 1 EPOCA
# =========================================================
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        out = model(xb, return_dict=True)
        logits = out["logits"]

        loss = F.binary_cross_entropy_with_logits(logits, yb)
        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / max(total_n, 1)


# =========================================================
# 6) FIT CON EARLY STOPPING
# =========================================================
def fit_model(
    model,
    train_loader,
    val_loader,
    device,
    lr=1e-3,
    weight_decay=1e-5,
    max_epochs=50,
    patience=10,
    use_scheduler=True
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=4
        )

    best_state = None
    best_bacc = -np.inf
    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = evaluate_model(model, val_loader, device)

        if scheduler is not None:
            scheduler.step(val_metrics["loss"])

        history.append({
            "epoch": epoch + 1,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_bacc": val_metrics["bacc"],
            "val_f1": val_metrics["f1"],
            "val_auc": val_metrics["auc"],
        })

        print(
            f"Epoch {epoch+1:03d}/{max_epochs:03d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_acc={val_metrics['acc']:.4f} | "
            f"val_bacc={val_metrics['bacc']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"val_auc={val_metrics['auc']:.4f}"
        )

        if val_metrics["bacc"] > best_bacc:
            best_bacc = val_metrics["bacc"]
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print(f"Early stopping en época {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_bacc, best_epoch

In [5]:
# =========================================================
# 7) CONSTRUIR EL MODELO HIBRIDO
# =========================================================
def build_hybrid_model(params):
    model = HybridTransformerTEKTE(
        chans=params["chans"],
        samples=params["samples"],

        d_model=params["d_model"],
        nhead=params["nhead"],
        num_transformer_layers=params["num_transformer_layers"],
        dim_feedforward=params["dim_feedforward"],
        transformer_dropout=params["transformer_dropout"],

        phi_kernel_size=params["phi_kernel_size"],
        dx=params["dx"],
        dy=params["dy"],
        tau=params["tau"],
        mu=params["mu"],

        kernel_type=params["kernel_type"],
        kernel_amplitude=params["kernel_amplitude"],
        kernel_length_scale=params["kernel_length_scale"],
        kernel_alpha=params["kernel_alpha"],
        trainable_kernel_amplitude=params["trainable_kernel_amplitude"],
        trainable_kernel_length_scale=params["trainable_kernel_length_scale"],
        trainable_kernel_alpha=params["trainable_kernel_alpha"],

        clf_hidden=params["clf_hidden"],
        clf_dropout=params["clf_dropout"],

        use_pre_transformer_norm=params["use_pre_transformer_norm"],
        use_post_transformer_norm=params["use_post_transformer_norm"],
    )
    return model


# =========================================================
# 8) OBJETIVO DE OPTUNA
#    AHORA USA train/val YA DEFINIDOS
# =========================================================
def make_objective(X_train, y_train, X_val, y_val, device, seed=42):
    def objective(trial):
        d_model = trial.suggest_categorical("d_model", [32, 64, 128])
        nhead = trial.suggest_categorical("nhead", [1, 2])

        if d_model % nhead != 0:
            raise optuna.exceptions.TrialPruned()

        params = {
            "chans": 19,
            "samples": 512,

            # Transformer
            "d_model": d_model,
            "nhead": nhead,
            "num_transformer_layers": trial.suggest_categorical("num_transformer_layers", [1,2]),
            "dim_feedforward": trial.suggest_categorical("dim_feedforward", [64, 128, 256]),
            "transformer_dropout": trial.suggest_categorical("transformer_dropout", [0.0, 0.1, 0.2, 0.3]),

            # Parte TE fija
            "phi_kernel_size": 64,
            "dx": 5,
            "dy": 5,
            "tau": 1,
            "mu": 4,

            # Kernel fijo
            "kernel_type": "rational_quadratic",
            "kernel_amplitude": 1.0,
            "kernel_length_scale": 1.0,
            "kernel_alpha": 1.0,
            "trainable_kernel_amplitude": False,
            "trainable_kernel_length_scale": False,
            "trainable_kernel_alpha": False,

            # Clasificador
            "clf_hidden": trial.suggest_categorical("clf_hidden", [32, 64, 128]),
            "clf_dropout": trial.suggest_categorical("clf_dropout", [0.0, 0.2, 0.3, 0.5]),

            # Norm
            "use_pre_transformer_norm": True,
            "use_post_transformer_norm": True,
        }

        lr = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

        train_ds = EEGDataset(X_train, y_train)
        val_ds = EEGDataset(X_val, y_val)

        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=False)
        val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, drop_last=False)

        model = build_hybrid_model(params).to(device)

        model, history, best_bacc, best_epoch = fit_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=30,
            patience=8,
            use_scheduler=True
        )

        val_metrics = evaluate_model(model, val_loader, device)

        trial.set_user_attr("best_epoch", int(best_epoch))
        trial.set_user_attr("params_full", params)
        trial.set_user_attr("lr", float(lr))
        trial.set_user_attr("weight_decay", float(weight_decay))

        return float(val_metrics["bacc"])

    return objective

In [6]:
class InspectableTransformerEncoder(nn.Module):
    """
    Réplica en idea del InspectableTransformerEncoder de TensorFlow/Keras.

    Flujo:
      input [B, T, C]
        -> proyección opcional a d_model
        -> MultiHeadAttention
        -> residual + LayerNorm
        -> FeedForward
        -> residual + LayerNorm

    Guarda los últimos attention scores para inspección.
    """
    def __init__(
        self,
        in_channels,
        d_model=None,
        num_heads=2,
        intermediate_dim=128,
        dropout=0.1,
        use_input_projection=True,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.d_model = d_model if d_model is not None else in_channels
        self.num_heads = num_heads
        self.intermediate_dim = intermediate_dim
        self.dropout_rate = dropout
        self.use_input_projection = use_input_projection

        if self.d_model % self.num_heads != 0:
            raise ValueError(
                f"d_model={self.d_model} debe ser divisible entre num_heads={self.num_heads}"
            )

        # Proyección opcional para imitar tu adaptación en PyTorch
        if use_input_projection or self.d_model != in_channels:
            self.input_proj = nn.Linear(in_channels, self.d_model)
        else:
            self.input_proj = nn.Identity()

        # Self-attention inspeccionable
        self.self_attention = nn.MultiheadAttention(
            embed_dim=self.d_model,
            num_heads=self.num_heads,
            dropout=self.dropout_rate,
            batch_first=True,
        )

        # Normalización y dropout, igual idea que Keras
        self.self_attention_layer_norm = nn.LayerNorm(self.d_model, eps=1e-6)
        self.self_attention_dropout = nn.Dropout(self.dropout_rate)

        # Feedforward
        self.feedforward_intermediate_dense = nn.Linear(self.d_model, self.intermediate_dim)
        self.feedforward_output_dense = nn.Linear(self.intermediate_dim, self.d_model)
        self.feedforward_dropout = nn.Dropout(self.dropout_rate)
        self.feedforward_layer_norm = nn.LayerNorm(self.d_model, eps=1e-6)

        # Para guardar los scores
        self._last_attention_scores = None

    def forward(self, inputs, training=False, need_weights=True, attn_mask=None):
        """
        inputs: [B, T, C]
        returns: [B, T, d_model]
        """
        x = self.input_proj(inputs)  # [B, T, d_model]

        attention_output, attention_scores = self.self_attention(
            query=x,
            key=x,
            value=x,
            attn_mask=attn_mask,
            need_weights=need_weights,
            average_attn_weights=False,  # devuelve [B, H, T, T]
        )

        if need_weights:
            self._last_attention_scores = attention_scores
        else:
            self._last_attention_scores = None

        attention_output = self.self_attention_dropout(attention_output)
        attention_output = self.self_attention_layer_norm(x + attention_output)

        ff_output = self.feedforward_intermediate_dense(attention_output)
        ff_output = F.gelu(ff_output)
        ff_output = self.feedforward_output_dense(ff_output)
        ff_output = self.feedforward_dropout(ff_output)

        output = self.feedforward_layer_norm(attention_output + ff_output)
        return output

    def get_attention_scores(self):
        if self._last_attention_scores is None:
            raise ValueError(
                "No se han calculado attention scores todavía. "
                "Haz un forward pass con need_weights=True."
            )
        return self._last_attention_scores

    def get_attention_weights(self):
        """
        Intenta imitar la idea de inspeccionar pesos Q, K, V y O.
        En PyTorch MultiheadAttention, Q/K/V suelen venir concatenados en in_proj_weight.
        """
        attn = self.self_attention

        weights_dict = {
            "in_proj_weight": attn.in_proj_weight.detach().cpu(),
            "in_proj_bias": attn.in_proj_bias.detach().cpu() if attn.in_proj_bias is not None else None,
            "out_proj_weight": attn.out_proj.weight.detach().cpu(),
            "out_proj_bias": attn.out_proj.bias.detach().cpu() if attn.out_proj.bias is not None else None,
        }
        return weights_dict

In [7]:
# =========================================================
# 3) FILTRO TEMPORAL POR CANAL  φ(.)
#    Réplica del bloque TF:
#    DepthwiseConv1D -> ReLU -> AvgPool1D -> BatchNorm1d
# =========================================================
class ChannelwiseTemporalFilter(nn.Module):
    def __init__(self, channels, kernel_size=9):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(
                in_channels=channels,
                out_channels=channels,
                kernel_size=kernel_size,
                stride=1,
                padding=0,          # TF: padding='valid'
                groups=channels,    # depthwise
                bias=True           # TF DepthwiseConv1D por defecto usa bias=True
            ),
            nn.ReLU(),
            nn.AvgPool1d(
                kernel_size=4,
                stride=4
            ),
            nn.BatchNorm1d(channels)
        )

    def forward(self, x):
        # x: [B, C, T]
        return self.net(x)

# =========================================================
# 4) TAKENS CONV1D
# =========================================================
class TakensConv1D(nn.Module):
    def __init__(self, dx=5, dy=5, tau=1, mu=4):
        super().__init__()
        self.dx = int(dx)
        self.dy = int(dy)
        self.tau = int(tau)
        self.mu = int(mu)
        self.num_filters = self.dx + self.dy + 1

        kernel_size = self.mu + (self.dx - 1) * self.tau + 1
        self.kernel_size = kernel_size

        kernel = torch.zeros(self.num_filters, 1, kernel_size)

        # x_sub_t_minus_mu
        for i in range(self.dx):
            kernel[i, 0, self.mu + i * self.tau] = 1.0

        # y_sub_t_minus_1
        for i in range(self.dy):
            kernel[self.dx + i, 0, (i + 1) * self.tau] = 1.0

        # y_sub_t
        kernel[self.dx + self.dy, 0, 0] = 1.0

        # Igual que en tu Keras: self.kernel = kernel.numpy()[::-1]
        kernel = torch.flip(kernel, dims=[-1])

        self.register_buffer("kernel", kernel)

    def forward(self, inputs):
        """
        inputs: [B, C, T]
        return:
            x_sub_t_minus_mu: [B, C, L, dx]
            y_sub_t_minus_1:  [B, C, L, dy]
            y_sub_t:          [B, C, L, 1]
        """
        B, C, T = inputs.shape

        reshaped = inputs.reshape(B * C, 1, T)

        conv_output = F.conv1d(
            reshaped,
            self.kernel,
            bias=None,
            stride=self.tau,
            padding=0
        )  # [B*C, num_filters, L]

        L = conv_output.shape[-1]
        output = conv_output.reshape(B, C, self.num_filters, L).permute(0, 1, 3, 2)

        x_sub_t_minus_mu = output[..., :self.dx]
        y_sub_t_minus_1 = output[..., self.dx:self.dx + self.dy]
        y_sub_t = output[..., -1:]

        return x_sub_t_minus_mu, y_sub_t_minus_1, y_sub_t


# =========================================================
# 5) KERNEL LAYER
#    Replica tu KernelLayer de TF
# =========================================================
class KernelLayer(nn.Module):
    def __init__(
        self,
        amplitude=1.0,
        trainable_amplitude=False,
        length_scale=1.0,
        trainable_length_scale=False,
        alpha=1.0,
        trainable_alpha=False,
        kernel_type="gaussian",
    ):
        super().__init__()
        self.kernel_type = kernel_type.lower()

        amp = torch.tensor(float(amplitude), dtype=torch.float32)
        ls = torch.tensor(float(length_scale), dtype=torch.float32)
        a = torch.tensor(float(alpha), dtype=torch.float32)

        if trainable_amplitude:
            self.amplitude = nn.Parameter(amp)
        else:
            self.register_buffer("amplitude", amp)

        if trainable_length_scale:
            self.length_scale = nn.Parameter(ls)
        else:
            self.register_buffer("length_scale", ls)

        if self.kernel_type == "rational_quadratic":
            if trainable_alpha:
                self.alpha = nn.Parameter(a)
            else:
                self.register_buffer("alpha", a)

    def forward(self, X):
        """
        X: [B, C, L, D]
        return: kernel.matrix(X, X) ~ [B, C, L, L]
        """
        diff = X.unsqueeze(-2) - X.unsqueeze(-3)   # [B, C, L, L, D]
        dist2 = (diff ** 2).sum(dim=-1)            # [B, C, L, L]

        amp = self.amplitude
        ls = torch.clamp(self.length_scale, min=1e-8)

        if self.kernel_type == "gaussian":
            K = (amp ** 2) * torch.exp(-dist2 / (2.0 * (ls ** 2)))
        elif self.kernel_type == "rational_quadratic":
            alpha = torch.clamp(self.alpha, min=1e-8)
            K = (amp ** 2) * (1.0 + dist2 / (2.0 * alpha * (ls ** 2))) ** (-alpha)
        else:
            raise ValueError(f"Unsupported kernel_type: {self.kernel_type}")

        return K


# =========================================================
# 6) TRANSFER ENTROPY LAYER
# =========================================================
class TransferEntropyLayer(nn.Module):
    def __init__(self, alpha=2):
        super().__init__()
        self.alpha = int(alpha)

    def compute_entropy(self, K_hadamard):
        """
        K_hadamard: [..., N, N]
        """
        trace_hadamard = torch.diagonal(K_hadamard, dim1=-2, dim2=-1).sum(dim=-1)
        trace_hadamard = trace_hadamard.unsqueeze(-1).unsqueeze(-1) + 1e-8

        K_normalized = K_hadamard / trace_hadamard

        K_power = K_normalized @ K_normalized
        trace_power = torch.diagonal(K_power, dim1=-2, dim2=-1).sum(dim=-1)

        if self.alpha == 2:
            H_alpha = -torch.log(trace_power + 1e-8)
        else:
            eigvals = torch.linalg.eigvalsh(K_normalized)
            eigvals = torch.clamp(eigvals.real, min=1e-8)
            H_alpha = torch.log((eigvals ** self.alpha).sum(dim=-1) + 1e-8) / (1 - self.alpha)

        return H_alpha

    def forward(self, K_x, K_y_minus_1, K_y):
        """
        K_x:         [B, C, L, L]
        K_y_minus_1: [B, C, L, L]
        K_y:         [B, C, L, L]

        return TE:   [B, C, C]
        """
        K_x_exp = K_x.unsqueeze(2)               # [B, C, 1, L, L]
        K_y_minus_1_exp = K_y_minus_1.unsqueeze(1)  # [B, 1, C, L, L]
        K_y_exp = K_y.unsqueeze(1)              # [B, 1, C, L, L]

        H_1 = self.compute_entropy(K_y_minus_1_exp * K_x_exp)
        H_2 = self.compute_entropy(K_y_exp * K_y_minus_1_exp * K_x_exp)
        H_3 = self.compute_entropy(K_y_exp * K_y_minus_1_exp)
        H_4 = self.compute_entropy(K_y_minus_1_exp)

        TE = H_1 - H_2 + H_3 - H_4
        return TE


# =========================================================
# 7) REMOVE DIAGONAL FLATTEN
# =========================================================
class RemoveDiagonalFlatten(nn.Module):
    def forward(self, inputs):
        """
        inputs: [B, C, C]
        return: [B, C*(C-1)]
        """
        B, C, C2 = inputs.shape
        if C != C2:
            raise ValueError("RemoveDiagonalFlatten: la matriz de entrada no es cuadrada.")

        mask = ~torch.eye(C, dtype=torch.bool, device=inputs.device)
        result = inputs[:, mask].reshape(B, C * (C - 1))
        return result

In [8]:
# =========================================================
# 8) MODELO HÍBRIDO Transformer + TEKTE
#    X^T -> Transformer -> HWp+bp -> φ -> Takens -> TE -> vec(T-diag(T)) -> clasificador
# =========================================================
class HybridTransformerTEKTE(nn.Module):
    def __init__(
        self,
        chans=19,
        samples=512,
        d_model=64,
        nhead=2,
        num_transformer_layers=1,   # se deja por compatibilidad, aunque el bloque actual usa 1 encoder manual
        dim_feedforward=128,
        transformer_dropout=0.1,
        phi_kernel_size=9,
        dx=5,
        dy=5,
        tau=1,
        mu=4,
        kernel_type="rational_quadratic",   # o "gaussian"
        kernel_amplitude=1.0,
        kernel_length_scale=1.0,
        kernel_alpha=1.0,
        trainable_kernel_amplitude=False,
        trainable_kernel_length_scale=False,
        trainable_kernel_alpha=False,
        clf_hidden=64,
        clf_dropout=0.3,
        use_pre_transformer_norm=True,
        use_post_transformer_norm=True,
    ):
        super().__init__()
        self.chans = chans
        self.samples = samples
        self.dx = dx
        self.dy = dy
        self.tau = tau
        self.mu = mu

        # 1) X: [B, C, T] -> [B, T, C]
        self.pre_transformer_norm = (
            nn.LayerNorm(chans) if use_pre_transformer_norm else nn.Identity()
        )

        # 2) Transformer
        self.transformer = InspectableTransformerEncoder(
            in_channels=chans,
            d_model=d_model,
            num_heads=nhead,
            intermediate_dim=dim_feedforward,
            dropout=transformer_dropout,
            use_input_projection=True,
        )

        self.post_transformer_norm = (
            nn.LayerNorm(d_model) if use_post_transformer_norm else nn.Identity()
        )

        # 3) projection back to channel space: Xf = H Wp + bp
        self.proj_back = nn.Linear(d_model, chans)

        # 4) φ(.)
        self.phi = ChannelwiseTemporalFilter(
            channels=chans,
            kernel_size=phi_kernel_size
        )

        # 5) Takens
        self.takens = TakensConv1D(dx=dx, dy=dy, tau=tau, mu=mu)

        # 6) Dense projections like your TF model
        self.dense_proj_x = nn.Linear(dx, dx, bias=False)
        self.dense_proj_y1 = nn.Linear(dy, dy, bias=False)
        self.dense_proj_y = nn.Linear(1, 1, bias=False)

        # 7) Kernel layers
        self.kernel_x = KernelLayer(
            amplitude=kernel_amplitude,
            trainable_amplitude=trainable_kernel_amplitude,
            length_scale=kernel_length_scale,
            trainable_length_scale=trainable_kernel_length_scale,
            alpha=kernel_alpha,
            trainable_alpha=trainable_kernel_alpha,
            kernel_type=kernel_type,
        )

        self.kernel_y_minus_1 = KernelLayer(
            amplitude=kernel_amplitude,
            trainable_amplitude=trainable_kernel_amplitude,
            length_scale=kernel_length_scale,
            trainable_length_scale=trainable_kernel_length_scale,
            alpha=kernel_alpha,
            trainable_alpha=trainable_kernel_alpha,
            kernel_type=kernel_type,
        )

        self.kernel_y = KernelLayer(
            amplitude=kernel_amplitude,
            trainable_amplitude=trainable_kernel_amplitude,
            length_scale=kernel_length_scale,
            trainable_length_scale=trainable_kernel_length_scale,
            alpha=kernel_alpha,
            trainable_alpha=trainable_kernel_alpha,
            kernel_type=kernel_type,
        )

        # 8) Transfer Entropy
        self.transfer_entropy = TransferEntropyLayer(alpha=2)

        # 9) remove diagonal
        self.remove_diag_flatten = RemoveDiagonalFlatten()

        # 10) classifier
        feat_dim = chans * (chans - 1)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, clf_hidden),
            nn.ReLU(),
            nn.Dropout(clf_dropout),
            nn.Linear(clf_hidden, 1)
        )

    def forward(self, x, return_dict=True, need_attention=False):
        """
        x: [B, C, T]
        """
        B, C, T = x.shape

        if C != self.chans:
            raise ValueError(f"Se esperaban {self.chans} canales y llegaron {C}.")
        if T != self.samples:
            raise ValueError(f"Se esperaban {self.samples} muestras y llegaron {T}.")

        # -------------------------------------------------
        # Hybrid Transformer–TEKTE math:
        # X^T -> Transformer -> H -> Xf = HWp + bp -> φ -> Takens -> TE -> classifier
        # -------------------------------------------------

        # (1) transpose: [B, C, T] -> [B, T, C]
        x_t = x.transpose(1, 2)

        # optional norm before transformer
        x_t = self.pre_transformer_norm(x_t)

        # (2) Transformer
        H = self.transformer(x_t, need_weights=need_attention)   # [B, T, d_model]

        # optional norm after transformer
        H = self.post_transformer_norm(H)

        # (3) projection back to channel space
        Xf = self.proj_back(H)                                   # [B, T, C]

        # back to [B, C, T]
        Xf = Xf.transpose(1, 2)

        # (4) nonlinear temporal filter φ
        phi = self.phi(Xf)

        # (5)-(7) Takens states
        x_sub, y_minus_1, y_t = self.takens(phi)

        # dense projections
        x_sub = self.dense_proj_x(x_sub)
        y_minus_1 = self.dense_proj_y1(y_minus_1)
        y_t = self.dense_proj_y(y_t)

        # kernels
        Kx = self.kernel_x(x_sub)
        Ky1 = self.kernel_y_minus_1(y_minus_1)
        Ky = self.kernel_y(y_t)

        # (8)-(10) Transfer Entropy matrix T
        Tmat = self.transfer_entropy(Kx, Ky1, Ky)                # [B, C, C]

        # (11) remove diagonal + flatten
        h = self.remove_diag_flatten(Tmat)

        # (12) classifier logits
        logits = self.classifier(h).squeeze(-1)
        probs = torch.sigmoid(logits)

        if return_dict:
            return {
                "logits": logits,
                "probs": probs,
                "T": Tmat,
                "features": h,
                "Xf": Xf,
                "phi": phi,
                "attention_scores": (
                    self.transformer.get_attention_scores()
                    if need_attention else None
                ),
            }

        return logits
# =========================================================
# 9) LOSS
# =========================================================
def classification_loss(logits, y_true):
    y_true = y_true.float()
    return F.binary_cross_entropy_with_logits(logits, y_true)

In [9]:
model = HybridTransformerTEKTE(
    chans=19,
    samples=512,
    d_model=64,
    nhead=2,
    num_transformer_layers=2,
    dim_feedforward=128,
    transformer_dropout=0.1,
    phi_kernel_size=64,
    dx=5,
    dy=5,
    tau=1,
    mu=4,
    kernel_type="rational_quadratic",   # o "gaussian"
    kernel_amplitude=1.0,
    kernel_length_scale=1.0,
    kernel_alpha=1.0,
    trainable_kernel_amplitude=False,
    trainable_kernel_length_scale=False,
    trainable_kernel_alpha=False,
    clf_hidden=64,
    clf_dropout=0.3,
).to(device)


class SummaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x, return_dict=True, need_attention=False)
        return out["logits"]


wrapped_model = SummaryWrapper(model).to(device)

summary(
    wrapped_model,
    input_size=(1, 19, 512),   # batch, chans, samples
    device=device,
    depth=4,
    col_names=("input_size", "output_size", "num_params", "trainable"),
)

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #                   Trainable
SummaryWrapper                                [1, 19, 512]              [1]                       --                        True
├─HybridTransformerTEKTE: 1-1                 [1, 19, 512]              --                        --                        True
│    └─LayerNorm: 2-1                         [1, 512, 19]              [1, 512, 19]              38                        True
│    └─InspectableTransformerEncoder: 2-2     [1, 512, 19]              [1, 512, 64]              --                        True
│    │    └─Linear: 3-1                       [1, 512, 19]              [1, 512, 64]              1,280                     True
│    │    └─MultiheadAttention: 3-2           --                        [1, 512, 64]              16,640                    True
│    │    └─Dropout: 3-3                      [1, 512, 64]              [1, 512, 64]        

In [10]:
# =========================================================
# 1) CARGA Y SEGMENTACIÓN DE LA BASE TDAH
# =========================================================
def segmentar_senales(db, labels, segmento_tamano=512, overlap=0.5):
    paso = int(segmento_tamano * (1 - overlap))
    if paso <= 0:
        raise ValueError("El paso debe ser mayor que 0.")

    segmentos = []
    y = []
    sbjs = []

    i = 0
    for sujeto, senal in db.items():
        C, T = senal.shape

        for inicio in range(0, T - segmento_tamano + 1, paso):
            segmento = senal[:, inicio:inicio + segmento_tamano]
            segmentos.append(segmento)
            y.append(labels[i])
            sbjs.append(sujeto)

        i += 1

    X = np.array(segmentos, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    sbjs = np.array(sbjs, dtype=object)

    return X, y, sbjs


def get_segmented_data_tdah():
    ruta_carpeta_tdah = "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/ADHD_group"
    ruta_carpeta_control = "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/Control_group"

    sujetos_tdah = sorted([archivo[:-4] for archivo in os.listdir(ruta_carpeta_tdah) if archivo.endswith(".mat")])
    sujetos_control = sorted([archivo[:-4] for archivo in os.listdir(ruta_carpeta_control) if archivo.endswith(".mat")])

    eeg_tdah = {}
    eeg_control = {}

    for sbj in sujetos_tdah:
        mat_file_path = os.path.join(ruta_carpeta_tdah, sbj + ".mat")
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_tdah[sbj] = data[columna].T.astype(np.float32)

    for sbj in sujetos_control:
        mat_file_path = os.path.join(ruta_carpeta_control, sbj + ".mat")
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_control[sbj] = data[columna].T.astype(np.float32)

    db = eeg_control | eeg_tdah

    zeros = np.zeros(len(eeg_control), dtype=np.int64)
    ones = np.ones(len(eeg_tdah), dtype=np.int64)
    labels = np.hstack((zeros, ones))

    X, y, sbjs = segmentar_senales(db, labels, segmento_tamano=512, overlap=0.5)

    print("Datos segmentados:")
    print("X.shape   =", X.shape)
    print("y.shape   =", y.shape)
    print("sbjs.shape=", sbjs.shape)

    return X, y, sbjs


def load_tdah_folds():
    folds_path = "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl"
    with open(folds_path, "rb") as f:
        folds = pickle.load(f)

    print("Número de folds:", len(folds))
    return folds


# =========================================================
# 2) UTILIDADES
# =========================================================
def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def extract_fold_subjects(fold_data):
    if isinstance(fold_data, (tuple, list)):
        if len(fold_data) == 3:
            train_subjects = np.array(fold_data[0], dtype=object)
            val_subjects   = np.array(fold_data[1], dtype=object)
            test_subjects  = np.array(fold_data[2], dtype=object)
            return train_subjects, val_subjects, test_subjects
        elif len(fold_data) == 2:
            train_subjects = np.array(fold_data[0], dtype=object)
            test_subjects  = np.array(fold_data[1], dtype=object)
            return train_subjects, None, test_subjects
        else:
            raise ValueError(f"Fold no reconocido. len={len(fold_data)}")

    elif isinstance(fold_data, dict):
        possible_train_keys = ["train_subjects", "train_sbjs", "train", "tr"]
        possible_val_keys   = ["val_subjects", "val_sbjs", "val", "va", "validation_subjects"]
        possible_test_keys  = ["test_subjects", "test_sbjs", "test", "te"]

        train_key = next((k for k in possible_train_keys if k in fold_data), None)
        val_key   = next((k for k in possible_val_keys if k in fold_data), None)
        test_key  = next((k for k in possible_test_keys if k in fold_data), None)

        if train_key is None or test_key is None:
            raise ValueError(f"No pude detectar sujetos train/test en fold_data={fold_data.keys()}")

        train_subjects = np.array(fold_data[train_key], dtype=object)
        val_subjects   = np.array(fold_data[val_key], dtype=object) if val_key is not None else None
        test_subjects  = np.array(fold_data[test_key], dtype=object)

        return train_subjects, val_subjects, test_subjects

    else:
        raise ValueError("Formato de folds.pkl no reconocido.")


def to_serializable(obj):
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    elif isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    else:
        return obj


def save_optuna_study(study, save_dir):
    ensure_dir(save_dir)

    # estudio completo en pickle
    with open(os.path.join(save_dir, "study.pkl"), "wb") as f:
        pickle.dump(study, f)

    # dataframe de trials
    df_trials = study.trials_dataframe()
    df_trials.to_csv(os.path.join(save_dir, "optuna_trials.csv"), index=False)

    # mejor trial en json
    best_info = {
        "best_value": float(study.best_value),
        "best_params": to_serializable(study.best_trial.params),
        "best_user_attrs": to_serializable(study.best_trial.user_attrs),
        "best_trial_number": int(study.best_trial.number),
    }
    with open(os.path.join(save_dir, "best_trial.json"), "w", encoding="utf-8") as f:
        json.dump(best_info, f, indent=4, ensure_ascii=False)


def save_training_history(history, save_path):
    df_hist = pd.DataFrame(history)
    df_hist.to_csv(save_path, index=False)


def save_test_metrics(test_metrics, save_path):
    metrics_to_save = to_serializable(test_metrics)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(metrics_to_save, f, indent=4, ensure_ascii=False)


def save_split_arrays(X_train, y_train, X_val, y_val, X_test, y_test, save_dir):
    ensure_dir(save_dir)

    np.save(os.path.join(save_dir, "X_train.npy"), X_train)
    np.save(os.path.join(save_dir, "y_train.npy"), y_train)
    np.save(os.path.join(save_dir, "X_val.npy"), X_val)
    np.save(os.path.join(save_dir, "y_val.npy"), y_val)
    np.save(os.path.join(save_dir, "X_test.npy"), X_test)
    np.save(os.path.join(save_dir, "y_test.npy"), y_test)


# =========================================================
# 3) OBJETIVO DE OPTUNA PARA UN SOLO FOLD
# =========================================================
def make_objective_for_fold(
    X_train,
    y_train,
    X_val,
    y_val,
    device,
    max_epochs=30,
    patience=8
):
    def objective(trial):
        d_model = trial.suggest_categorical("d_model", [32, 64, 128])
        nhead = trial.suggest_categorical("nhead", [1, 2, 4])

        if d_model % nhead != 0:
            raise optuna.exceptions.TrialPruned()

        params = {
            "chans": 19,
            "samples": 512,
            "d_model": d_model,
            "nhead": nhead,
            "num_transformer_layers": trial.suggest_categorical("num_transformer_layers", [1]),
            "dim_feedforward": trial.suggest_categorical("dim_feedforward", [64, 128, 256]),
            "transformer_dropout": trial.suggest_categorical("transformer_dropout", [0.0, 0.1, 0.2, 0.3]),
            "phi_kernel_size": 64,
            "dx": 5,
            "dy": 5,
            "tau": 1,
            "mu": 4,
            "kernel_type": "rational_quadratic",
            "kernel_amplitude": 1.0,
            "kernel_length_scale": 1.0,
            "kernel_alpha": 1.0,
            "trainable_kernel_amplitude": False,
            "trainable_kernel_length_scale": False,
            "trainable_kernel_alpha": False,
            "clf_hidden": trial.suggest_categorical("clf_hidden", [32, 64, 128]),
            "clf_dropout": trial.suggest_categorical("clf_dropout", [0.0, 0.2, 0.3, 0.5]),
            "use_pre_transformer_norm": True,
            "use_post_transformer_norm": True,
        }

        lr = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

        train_ds = EEGDataset(X_train, y_train)
        val_ds   = EEGDataset(X_val, y_val)

        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=False)
        val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, drop_last=False)

        model = build_hybrid_model(params).to(device)

        model, history, best_bacc, best_epoch = fit_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            use_scheduler=True
        )

        val_metrics = evaluate_model(model, val_loader, device)

        trial.set_user_attr("best_epoch", int(best_epoch))
        trial.set_user_attr("params_full", params)
        trial.set_user_attr("lr", float(lr))
        trial.set_user_attr("weight_decay", float(weight_decay))

        return float(val_metrics["bacc"])

    return objective


# =========================================================
# 4) OPTUNA EN UN SOLO FOLD
# =========================================================
def run_optuna_on_fold(
    X,
    y,
    sbjs,
    folds,
    fold_id,
    n_trials=5,
    seed=42,
    max_epochs=30,
    patience=8,
    save_root="results_tdah"
):
    seed_everything(seed)
    device = get_device()

    fold_dir = os.path.join(save_root, f"fold_{fold_id+1}")
    ensure_dir(fold_dir)

    train_subjects, val_subjects, test_subjects = extract_fold_subjects(folds[fold_id])

    if val_subjects is None:
        raise ValueError("Este código espera folds con train/val/test.")

    train_idx = np.array([i for i, s in enumerate(sbjs) if s in train_subjects])
    val_idx   = np.array([i for i, s in enumerate(sbjs) if s in val_subjects])
    test_idx  = np.array([i for i, s in enumerate(sbjs) if s in test_subjects])

    X_train, y_train = X[train_idx], y[train_idx]
    X_val,   y_val   = X[val_idx], y[val_idx]
    X_test,  y_test  = X[test_idx], y[test_idx]

    print("\n" + "=" * 80)
    print(f"FOLD {fold_id+1}")
    print("=" * 80)
    print(f"Train: {X_train.shape}")
    print(f"Val  : {X_val.shape}")
    print(f"Test : {X_test.shape}")
    print(f"Guardando en: {fold_dir}")

    # guardar particiones
    save_split_arrays(X_train, y_train, X_val, y_val, X_test, y_test, os.path.join(fold_dir, "splits"))

    # guardar sujetos del fold
    split_subjects = {
        "train_subjects": train_subjects.tolist(),
        "val_subjects": val_subjects.tolist(),
        "test_subjects": test_subjects.tolist(),
    }
    with open(os.path.join(fold_dir, "subjects_split.json"), "w", encoding="utf-8") as f:
        json.dump(split_subjects, f, indent=4, ensure_ascii=False)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed)
    )

    objective = make_objective_for_fold(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        device=device,
        max_epochs=max_epochs,
        patience=patience
    )

    study.optimize(objective, n_trials=n_trials)

    print("\nMejor trial:")
    print("Best value:", study.best_value)
    print("Best params:", study.best_trial.params)

    save_optuna_study(study, os.path.join(fold_dir, "optuna"))

    return study, X_train, y_train, X_val, y_val, X_test, y_test, device, fold_dir


# =========================================================
# 5) ENTRENAMIENTO FINAL Y TEST
# =========================================================
def train_and_test_best_model_on_fold(
    study,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    device,
    fold_dir,
    final_max_epochs=100,
    final_patience=10
):
    best_params = study.best_trial.user_attrs["params_full"]
    lr = study.best_trial.user_attrs["lr"]
    weight_decay = study.best_trial.user_attrs["weight_decay"]

    X_trainval = np.concatenate([X_train, X_val], axis=0)
    y_trainval = np.concatenate([y_train, y_val], axis=0)

    trainval_ds = EEGDataset(X_trainval, y_trainval)
    val_ds      = EEGDataset(X_val, y_val)
    test_ds     = EEGDataset(X_test, y_test)

    trainval_loader = DataLoader(trainval_ds, batch_size=32, shuffle=True, drop_last=False)
    val_loader      = DataLoader(val_ds, batch_size=64, shuffle=False, drop_last=False)
    test_loader     = DataLoader(test_ds, batch_size=64, shuffle=False, drop_last=False)

    model = build_hybrid_model(best_params).to(device)

    model, history, best_bacc, best_epoch = fit_model(
        model=model,
        train_loader=trainval_loader,
        val_loader=val_loader,
        device=device,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=final_max_epochs,
        patience=final_patience,
        use_scheduler=True
    )

    test_metrics = evaluate_model(model, test_loader, device)

    print("\n" + "=" * 80)
    print("RESULTADOS EN TEST")
    print("=" * 80)
    print(f"Acc : {test_metrics['acc']:.4f}")
    print(f"BAcc: {test_metrics['bacc']:.4f}")
    print(f"F1  : {test_metrics['f1']:.4f}")
    print(f"AUC : {test_metrics['auc']:.4f}")
    print("CM:")
    print(test_metrics["cm"])

    # -----------------------------
    # GUARDADOS
    # -----------------------------
    final_dir = os.path.join(fold_dir, "final_model")
    ensure_dir(final_dir)

    # pesos del modelo
    torch.save(model.state_dict(), os.path.join(final_dir, "best_model_state_dict.pt"))

    # checkpoint más completo
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "best_params": best_params,
        "lr": lr,
        "weight_decay": weight_decay,
        "best_epoch": int(best_epoch),
        "best_bacc": float(best_bacc),
    }
    torch.save(checkpoint, os.path.join(final_dir, "checkpoint_complete.pt"))

    # arquitectura y mejores hp
    with open(os.path.join(final_dir, "best_params.json"), "w", encoding="utf-8") as f:
        json.dump(to_serializable(best_params), f, indent=4, ensure_ascii=False)

    # historial
    save_training_history(history, os.path.join(final_dir, "history.csv"))

    # métricas test
    save_test_metrics(test_metrics, os.path.join(final_dir, "test_metrics.json"))

    return study, model, history, test_metrics


# =========================================================
# 6) PIPELINE COMPLETO SOLO FOLD 1
# =========================================================
SAVE_ROOT = "/home/usuario-utp/Desktop/resultados_tdah_hybrid_fold1"
ensure_dir(SAVE_ROOT)

folds = load_tdah_folds()
X, y, sbjs = get_segmented_data_tdah()

# guardar datos globales también
np.save(os.path.join(SAVE_ROOT, "X_all.npy"), X)
np.save(os.path.join(SAVE_ROOT, "y_all.npy"), y)
np.save(os.path.join(SAVE_ROOT, "sbjs_all.npy"), sbjs)

study, X_train, y_train, X_val, y_val, X_test, y_test, device, fold_dir = run_optuna_on_fold(
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    fold_id=0,          # fold 1
    n_trials=6,
    seed=42,
    max_epochs=100,
    patience=8,
    save_root=SAVE_ROOT
)

study, model, history, test_metrics = train_and_test_best_model_on_fold(
    study=study,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
    device=device,
    fold_dir=fold_dir,
    final_max_epochs=100,
    final_patience=10
)

# resumen final
summary = {
    "fold_id": 0,
    "device": device,
    "best_optuna_value": float(study.best_value),
    "best_optuna_params": to_serializable(study.best_trial.params),
    "test_metrics": to_serializable(test_metrics),
    "save_root": SAVE_ROOT,
    "fold_dir": fold_dir,
}

with open(os.path.join(SAVE_ROOT, "run_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("\n" + "=" * 80)
print("EJECUCIÓN FINALIZADA")
print("=" * 80)
print(f"Resultados guardados en: {SAVE_ROOT}")
print(f"Resultados del fold en: {fold_dir}")

Número de folds: 5
Datos segmentados:
X.shape   = (8279, 19, 512)
y.shape   = (8279,)
sbjs.shape= (8279,)


[I 2026-04-10 14:24:47,806] A new study created in memory with name: no-name-1fad365a-4efd-4fff-8e49-16174fa6e20a



FOLD 1
Train: (4977, 19, 512)
Val  : (1574, 19, 512)
Test : (1662, 19, 512)
Guardando en: /home/usuario-utp/Desktop/resultados_tdah_hybrid_fold1/fold_1
Epoch 001/100 | train_loss=0.4983 | val_loss=0.4678 | val_acc=0.8240 | val_bacc=0.8056 | val_f1=0.8587 | val_auc=0.9135
Epoch 002/100 | train_loss=0.2531 | val_loss=0.4136 | val_acc=0.8050 | val_bacc=0.7956 | val_f1=0.8341 | val_auc=0.9099
Epoch 003/100 | train_loss=0.0827 | val_loss=0.8899 | val_acc=0.7230 | val_bacc=0.7077 | val_f1=0.7710 | val_auc=0.8523
Epoch 004/100 | train_loss=0.0341 | val_loss=0.7218 | val_acc=0.7986 | val_bacc=0.7987 | val_f1=0.8177 | val_auc=0.8774
Epoch 005/100 | train_loss=0.0217 | val_loss=0.8655 | val_acc=0.7935 | val_bacc=0.7988 | val_f1=0.8062 | val_auc=0.8472
Epoch 006/100 | train_loss=0.0146 | val_loss=0.9940 | val_acc=0.7522 | val_bacc=0.7381 | val_f1=0.7943 | val_auc=0.8808
Epoch 007/100 | train_loss=0.0101 | val_loss=1.2951 | val_acc=0.7338 | val_bacc=0.7225 | val_f1=0.7746 | val_auc=0.8467
Epoch 0

[I 2026-04-10 14:29:13,682] Trial 0 finished with value: 0.8055921176955828 and parameters: {'d_model': 64, 'nhead': 1, 'num_transformer_layers': 1, 'dim_feedforward': 128, 'transformer_dropout': 0.2, 'clf_hidden': 32, 'clf_dropout': 0.2, 'learning_rate': 0.0008012737503998541, 'weight_decay': 2.62108787826544e-06}. Best is trial 0 with value: 0.8055921176955828.


Epoch 001/100 | train_loss=0.6912 | val_loss=0.6823 | val_acc=0.5661 | val_bacc=0.5000 | val_f1=0.7229 | val_auc=0.9226
Epoch 002/100 | train_loss=0.6510 | val_loss=0.5795 | val_acc=0.5661 | val_bacc=0.5000 | val_f1=0.7229 | val_auc=0.8910
Epoch 003/100 | train_loss=0.4820 | val_loss=0.5475 | val_acc=0.6792 | val_bacc=0.6459 | val_f1=0.7601 | val_auc=0.8354
Epoch 004/100 | train_loss=0.3872 | val_loss=0.5885 | val_acc=0.6868 | val_bacc=0.6658 | val_f1=0.7489 | val_auc=0.8020
Epoch 005/100 | train_loss=0.3487 | val_loss=0.5449 | val_acc=0.7370 | val_bacc=0.7299 | val_f1=0.7713 | val_auc=0.7902
Epoch 006/100 | train_loss=0.3215 | val_loss=0.5689 | val_acc=0.7446 | val_bacc=0.7394 | val_f1=0.7754 | val_auc=0.7847
Epoch 007/100 | train_loss=0.2946 | val_loss=0.6065 | val_acc=0.7287 | val_bacc=0.7206 | val_f1=0.7655 | val_auc=0.7760
Epoch 008/100 | train_loss=0.2672 | val_loss=0.6347 | val_acc=0.7370 | val_bacc=0.7395 | val_f1=0.7562 | val_auc=0.7555
Epoch 009/100 | train_loss=0.2453 | val_

...

Epoch 001/100 | train_loss=0.6766 | val_loss=0.6448 | val_acc=0.5438 | val_bacc=0.4807 | val_f1=0.7040 | val_auc=0.8059
Epoch 002/100 | train_loss=0.4721 | val_loss=0.5912 | val_acc=0.6607 | val_bacc=0.6279 | val_f1=0.7452 | val_auc=0.7928
Epoch 003/100 | train_loss=0.2516 | val_loss=0.6388 | val_acc=0.6912 | val_bacc=0.6685 | val_f1=0.7550 | val_auc=0.7932
Epoch 004/100 | train_loss=0.1232 | val_loss=0.7260 | val_acc=0.6912 | val_bacc=0.6803 | val_f1=0.7367 | val_auc=0.8000
Epoch 005/100 | train_loss=0.0686 | val_loss=0.9948 | val_acc=0.6309 | val_bacc=0.6278 | val_f1=0.6663 | val_auc=0.7540
Epoch 006/100 | train_loss=0.0380 | val_loss=1.0920 | val_acc=0.6309 | val_bacc=0.6318 | val_f1=0.6572 | val_auc=0.7561
Epoch 007/100 | train_loss=0.0280 | val_loss=1.0780 | val_acc=0.6753 | val_bacc=0.6715 | val_f1=0.7095 | val_auc=0.7796
Epoch 008/100 | train_loss=0.0213 | val_loss=1.2918 | val_acc=0.6328 | val_bacc=0.6382 | val_f1=0.6480 | val_auc=0.7552
Epoch 009/100 | train_loss=0.0180 | val_

[I 2026-04-10 15:03:50,678] Trial 4 finished with value: 0.6802759989680438 and parameters: {'d_model': 32, 'nhead': 2, 'num_transformer_layers': 1, 'dim_feedforward': 64, 'transformer_dropout': 0.0, 'clf_hidden': 32, 'clf_dropout': 0.0, 'learning_rate': 0.00021775224101934097, 'weight_decay': 1.7019223026554036e-06}. Best is trial 3 with value: 0.8787574788062831.


Epoch 001/100 | train_loss=0.4570 | val_loss=0.5578 | val_acc=0.7560 | val_bacc=0.7773 | val_f1=0.7409 | val_auc=0.8304
Epoch 002/100 | train_loss=0.0691 | val_loss=0.8437 | val_acc=0.7446 | val_bacc=0.7331 | val_f1=0.7843 | val_auc=0.8573
Epoch 003/100 | train_loss=0.0299 | val_loss=0.7346 | val_acc=0.8100 | val_bacc=0.8119 | val_f1=0.8263 | val_auc=0.8983
Epoch 004/100 | train_loss=0.0201 | val_loss=0.8087 | val_acc=0.7903 | val_bacc=0.7878 | val_f1=0.8133 | val_auc=0.8936
Epoch 005/100 | train_loss=0.0184 | val_loss=0.9285 | val_acc=0.7929 | val_bacc=0.7998 | val_f1=0.8034 | val_auc=0.8745
Epoch 006/100 | train_loss=0.0214 | val_loss=0.6303 | val_acc=0.8348 | val_bacc=0.8442 | val_f1=0.8413 | val_auc=0.9319
Epoch 007/100 | train_loss=0.0063 | val_loss=0.6812 | val_acc=0.8355 | val_bacc=0.8439 | val_f1=0.8429 | val_auc=0.9283
Epoch 008/100 | train_loss=0.0031 | val_loss=0.7845 | val_acc=0.8259 | val_bacc=0.8297 | val_f1=0.8390 | val_auc=0.9124
Epoch 009/100 | train_loss=0.0025 | val_

[I 2026-04-10 15:10:50,496] Trial 5 finished with value: 0.8441844835207452 and parameters: {'d_model': 128, 'nhead': 4, 'num_transformer_layers': 1, 'dim_feedforward': 256, 'transformer_dropout': 0.2, 'clf_hidden': 128, 'clf_dropout': 0.2, 'learning_rate': 0.00041358679552638914, 'weight_decay': 4.637921903458029e-06}. Best is trial 3 with value: 0.8787574788062831.



Mejor trial:
Best value: 0.8787574788062831
Best params: {'d_model': 64, 'nhead': 1, 'num_transformer_layers': 1, 'dim_feedforward': 128, 'transformer_dropout': 0.1, 'clf_hidden': 64, 'clf_dropout': 0.0, 'learning_rate': 0.001195960383019184, 'weight_decay': 8.178476574339548e-05}
Epoch 001/100 | train_loss=0.2777 | val_loss=0.1310 | val_acc=0.9409 | val_bacc=0.9321 | val_f1=0.9503 | val_auc=0.9984
Epoch 002/100 | train_loss=0.0554 | val_loss=0.0222 | val_acc=0.9956 | val_bacc=0.9952 | val_f1=0.9961 | val_auc=0.9999
Epoch 003/100 | train_loss=0.0257 | val_loss=0.0125 | val_acc=0.9962 | val_bacc=0.9966 | val_f1=0.9966 | val_auc=1.0000
Epoch 004/100 | train_loss=0.0263 | val_loss=0.0059 | val_acc=0.9987 | val_bacc=0.9989 | val_f1=0.9989 | val_auc=1.0000
Epoch 005/100 | train_loss=0.0122 | val_loss=0.0119 | val_acc=0.9975 | val_bacc=0.9971 | val_f1=0.9978 | val_auc=1.0000
Epoch 006/100 | train_loss=0.0110 | val_loss=0.0043 | val_acc=0.9987 | val_bacc=0.9985 | val_f1=0.9989 | val_auc=1.00